# 00b - Base Label Playground

Exports a small random sample (e.g., 500 tweets) from `base_dataset.pkl` to a CSV.
Use this file for exploratory hand-labelling to get a feel for the data before
formalizing categories or criteria.

In [ ]:
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

In [ ]:
import pandas as pd
from pathlib import Path

## Configuration

In [ ]:
SAMPLE_SIZE = 500
INPUT_PATH  = hitl_folder / 'base_dataset.pkl'
OUTPUT_CSV  = hitl_folder / 'base_playground_sample.csv'

## Load and Sample

In [ ]:
assert INPUT_PATH.exists(), f'Input not found: {INPUT_PATH}. Run 00_hitl_data_preparation.ipynb first.'
df = pd.read_pickle(INPUT_PATH)
print(f'Loaded {len(df):,} tweets from {INPUT_PATH.name}')

sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).copy()
print(f'Sampled {len(sample_df)} tweets.')

## Export to CSV

The CSV uses the standard schema so it can be easily inspected or dropped into the pipeline later if desired.

In [ ]:
if 'text' in sample_df.columns:
    sample_df['text'] = sample_df['text'].astype(str).str.replace('\n', ' ', regex=False)

hitl_schema_cols = ['id', 'text', 'processed_text', 'type', 'likes', 'retweets', 'predicted_label', 'human_label']
for col in hitl_schema_cols:
    if col not in sample_df.columns:
        sample_df[col] = ''

sample_df[hitl_schema_cols].to_csv(OUTPUT_CSV, index=False)
print(f'Saved → {OUTPUT_CSV}')